<a href="https://colab.research.google.com/github/Razae786/NLP-Lab-FA23-BAI-064/blob/main/FA23_BAI_064_Hafiz_Raza_E_Mustafa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AIC365 - Natural Language Processing (Lab Mid Exam)
**Name:** Hafiz Raza E Mustafa  
**Roll No:** FA23-BAI-064  
**Class:** BSAI-7  
Sir Zaeem Akbar Khan

## Question 1: Customer Review Mining for an Online Electronics Store (5 Marks)

In [12]:
from bs4 import BeautifulSoup
from collections import Counter, defaultdict
import pandas as pd

html_doc = """
<div class="review" data-product="Wireless Earbuds X2">
<span class="rating">5</span>
<p class="text">Superb sound quality!! Battery lasts all day.</p></div>
<div class="review" data-product="Wireless Earbuds X2">
<span class="rating">2</span>
<p class="text">Left earbud stopped working after a week.</p></div>
<div class="review" data-product="Smart Watch S5">
<span class="rating">4</span>
<p class="text">Good display, strap could be better.</p></div>
<div class="review" data-product="Smart Watch S5">
<p class="text">Delivery was late but the watch is nice.</p></div>
<div class="review" data-product="Wireless Earbuds X2">
<span class="rating">4</span>
<p class="text">Value for money. Delivery was quick.</p></div>
<div class="review" data-product="Smart Watch S5">
<span class="rating">1</span>
<p class="text">Screen cracked on arrival. Very disappointed.</p></div>
"""

soup = BeautifulSoup(html_doc, "html.parser")
reviews = []

for div in soup.find_all("div", class_="review"):
    product = div.get("data-product")

    rating_tag = div.find("span", class_="rating")
    if rating_tag is not None:
        rating = int(rating_tag.text.strip())
    else:
        rating = None

    text = div.find("p", class_="text").text.strip()

    reviews.append({
        "product": product,
        "rating": rating,
        "text": text
    })

print("Extracted Reviews:")
for r in reviews:
    print(r)

Extracted Reviews:
{'product': 'Wireless Earbuds X2', 'rating': 5, 'text': 'Superb sound quality!! Battery lasts all day.'}
{'product': 'Wireless Earbuds X2', 'rating': 2, 'text': 'Left earbud stopped working after a week.'}
{'product': 'Smart Watch S5', 'rating': 4, 'text': 'Good display, strap could be better.'}
{'product': 'Smart Watch S5', 'rating': None, 'text': 'Delivery was late but the watch is nice.'}
{'product': 'Wireless Earbuds X2', 'rating': 4, 'text': 'Value for money. Delivery was quick.'}
{'product': 'Smart Watch S5', 'rating': 1, 'text': 'Screen cracked on arrival. Very disappointed.'}


# **Part 2**

In [13]:
review_counts = Counter(r["product"] for r in reviews)
print("(i) Number of reviews per product:")
for prod, count in review_counts.items():
    print(f"  {prod}: {count}")

ratings_by_prod = defaultdict(list)
for r in reviews:
    if r["rating"] is not None:
        ratings_by_prod[r["product"]].append(r["rating"])

print("\n(ii) Average rating per product:")
for prod, vals in ratings_by_prod.items():
    avg = sum(vals) / len(vals)
    print(f"  {prod}: {avg:.2f}")

print("\n(iii) Reviews mentioning 'delivery':")
for r in reviews:
    if "delivery" in r["text"].lower():
        print(f"  - {r['text']}")

(i) Number of reviews per product:
  Wireless Earbuds X2: 3
  Smart Watch S5: 3

(ii) Average rating per product:
  Wireless Earbuds X2: 3.67
  Smart Watch S5: 2.50

(iii) Reviews mentioning 'delivery':
  - Delivery was late but the watch is nice.
  - Value for money. Delivery was quick.


# **Part 3**

In [14]:
df = pd.DataFrame(reviews)
df_clean = df.dropna(subset=["rating"])

df_clean.to_csv("reviews_clean.csv", index=False)

df_reloaded = pd.read_csv("reviews_clean.csv")
print("Shape:", df_reloaded.shape)
print("\nFirst Rows:")
print(df_reloaded.head())

Shape: (5, 3)

First Rows:
               product  rating                                           text
0  Wireless Earbuds X2     5.0  Superb sound quality!! Battery lasts all day.
1  Wireless Earbuds X2     2.0      Left earbud stopped working after a week.
2       Smart Watch S5     4.0           Good display, strap could be better.
3  Wireless Earbuds X2     4.0           Value for money. Delivery was quick.
4       Smart Watch S5     1.0  Screen cracked on arrival. Very disappointed.


## Question 2: Complaint Analysis for a Telecom Customer-Care Centre (10 Marks)

In [15]:
import re

complaints = [
    "@TelcoCare my 4G internet is NOT working since 2 days!!! Call me on 0300-1234567 "
    "or mail ali.raza@gmail.com. #NetworkDown https://t.co/Xy12Ab",
    "Dr. Sana here. The bills were wrongly charged in Lahore. Nobody is listening to "
    "customers! Contact: 0321-7654321 #BillingIssue"
]

for i, comp in enumerate(complaints, 1):
    info = {
        "mobile_numbers": re.findall(r"\b03\d{2}-\d{7}\b", comp),
        "emails": re.findall(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", comp),
        "hashtags": re.findall(r"#\w+", comp),
        "urls": re.findall(r"https?://\S+", comp),
        "mentions": re.findall(r"(?<!\w)@\w+", comp)
    }
    print(f"Complaint {i}:")
    print(info)

Complaint 1:
{'mobile_numbers': ['0300-1234567'], 'emails': ['ali.raza@gmail.com'], 'hashtags': ['#NetworkDown'], 'urls': ['https://t.co/Xy12Ab'], 'mentions': ['@TelcoCare']}
Complaint 2:
{'mobile_numbers': ['0321-7654321'], 'emails': [], 'hashtags': ['#BillingIssue'], 'urls': [], 'mentions': []}


# Part **2**

In [16]:
import re

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

stop_words = set(stopwords.words("english")) - {"not", "no", "nor", "never"}

def preprocess(text):
    sentences = sent_tokenize(text)
    cleaned = []

    for s in sentences:
        s = re.sub(r"https?://\S+", "", s)
        s = re.sub(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", "", s)
        s = re.sub(r"(?<!\w)@\w+", "", s)
        s = re.sub(r"\b03\d{2}-\d{7}\b", "", s)

        s = re.sub(r"#(\w+)", r"\1", s)

        s = s.lower()
        s = re.sub(r"[^\w\s]", " ", s)
        s = re.sub(r"\d+", " ", s)

        tokens = [w for w in word_tokenize(s) if w not in stop_words]
        if tokens:
            cleaned.append(tokens)

    return cleaned

for i, comp in enumerate(complaints, 1):
    print(f"Complaint {i} Sentence Tokens:")
    for sent_tokens in preprocess(comp):
        print(" ", sent_tokens)

Complaint 1 Sentence Tokens:
  ['g', 'internet', 'not', 'working', 'since', 'days']
  ['call', 'mail']
  ['networkdown']
Complaint 2 Sentence Tokens:
  ['dr', 'sana']
  ['bills', 'wrongly', 'charged', 'lahore']
  ['nobody', 'listening', 'customers']
  ['contact', 'billingissue']


***Part 3***

In [17]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

words = ["connected", "connecting", "charges", "was", "better", "studies"]
pos_tags = {
    "connected": "v",
    "connecting": "v",
    "charges": "n",
    "was": "v",
    "better": "a",
    "studies": "n"
}

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

print(f"{'Word':<12} {'POS':<6} {'PorterStemmer':<16} {'WordNetLemmatizer':<18}")
print("-" * 54)
for w in words:
    stem_w = stemmer.stem(w)
    lemma_w = lemmatizer.lemmatize(w, pos=pos_tags[w])
    print(f"{w:<12} {pos_tags[w]:<6} {stem_w:<16} {lemma_w:<18}")

Word         POS    PorterStemmer    WordNetLemmatizer 
------------------------------------------------------
connected    v      connect          connect           
connecting   v      connect          connect           
charges      n      charg            charge            
was          v      wa               be                
better       a      better           good              
studies      n      studi            study             


# **Part 4**

In [18]:
def min_edit_distance(source, target):
    s = source.lower()
    t = target.lower()
    m, n = len(s), len(t)

    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            sub_cost = 0 if s[i - 1] == t[j - 1] else 2
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + sub_cost
            )
    return dp[m][n]

def correct_city(word, cities):
    best_city = min(cities, key=lambda c: min_edit_distance(word, c))
    return best_city, min_edit_distance(word, best_city)

cities = ["Islamabad", "Lahore", "Karachi", "Peshawar", "Quetta"]
typed = ["Islamabd", "Lahor", "Karchi"]

for w in typed:
    city, dist = correct_city(w, cities)
    print(f"Typed: {w:<10} -> Closest City: {city:<10} (Distance: {dist})")

Typed: Islamabd   -> Closest City: Islamabad  (Distance: 1)
Typed: Lahor      -> Closest City: Lahore     (Distance: 1)
Typed: Karchi     -> Closest City: Karachi    (Distance: 1)


## Question 3: Autocomplete and Sentiment Detection for a Food Delivery Startup (10 Marks)

In [19]:
import math
from collections import Counter

corpus = [
    "<s> the food was delicious </s>",
    "<s> the food was cold </s>",
    "<s> the delivery was late </s>",
    "<s> the delivery was late </s>",
    "<s> the delivery was fast </s>",
]

unigrams = Counter()
bigrams = Counter()

for line in corpus:
    tokens = line.split()
    for t in tokens:
        unigrams[t] += 1
    for i in range(len(tokens) - 1):
        bigrams[(tokens[i], tokens[i + 1])] += 1

vocab = set(unigrams.keys()) - {"<s>"}
V = len(vocab)

print("(i) Unigram Counts:", dict(unigrams))
print("(i) Bigram Counts:", dict(bigrams))
print("Vocabulary Size V:", V)

def laplace_prob(w_prev, w_curr):
    return (bigrams[(w_prev, w_curr)] + 1) / (unigrams[w_prev] + V)

unsmoothed_p = bigrams[("food", "late")] / unigrams["food"]
smoothed_p = laplace_prob("food", "late")
print(f"\n(iii) Unsmoothed P(late | food) = {unsmoothed_p:.4f}")
print(f"(iii) Smoothed P(late | food)   = {smoothed_p:.4f}")

def sentence_log_prob(sentence):
    tokens = ["<s>"] + sentence.split() + ["</s>"]
    log_p = 0.0
    for i in range(len(tokens) - 1):
        log_p += math.log(laplace_prob(tokens[i], tokens[i + 1]))
    return log_p

s1 = "the food was late"
s2 = "the delivery was late"
lp1 = sentence_log_prob(s1)
lp2 = sentence_log_prob(s2)

print(f"\n(iv) Smoothed Log-Prob ('{s1}'): {lp1:.4f}")
print(f"(iv) Smoothed Log-Prob ('{s2}'): {lp2:.4f}")

(i) Unigram Counts: {'<s>': 5, 'the': 5, 'food': 2, 'was': 5, 'delicious': 1, '</s>': 5, 'cold': 1, 'delivery': 3, 'late': 2, 'fast': 1}
(i) Bigram Counts: {('<s>', 'the'): 5, ('the', 'food'): 2, ('food', 'was'): 2, ('was', 'delicious'): 1, ('delicious', '</s>'): 1, ('was', 'cold'): 1, ('cold', '</s>'): 1, ('the', 'delivery'): 3, ('delivery', 'was'): 3, ('was', 'late'): 2, ('late', '</s>'): 2, ('was', 'fast'): 1, ('fast', '</s>'): 1}
Vocabulary Size V: 9

(iii) Unsmoothed P(late | food) = 0.0000
(iii) Smoothed P(late | food)   = 0.0909

(iv) Smoothed Log-Prob ('the food was late'): -6.5268
(iv) Smoothed Log-Prob ('the delivery was late'): -6.0384


# **Part 2**

In [20]:
import math
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

train = [
    ("The food was delicious and fresh", "Positive"),
    ("Very fast delivery, loved it", "Positive"),
    ("Great taste and good packaging", "Positive"),
    ("Excellent service, will order again", "Positive"),
    ("The food was cold and stale", "Negative"),
    ("Delivery was very late", "Negative"),
    ("Terrible taste, waste of money", "Negative"),
    ("Worst service, rider was rude", "Negative")
]

test = [
    ("Fresh food and fast delivery", "Positive"),
    ("Late delivery and cold food", "Negative"),
    ("Excellent taste, will order again", "Positive"),
    ("Rude rider and terrible service", "Negative")
]

X_train_raw = [x[0] for x in train]
y_train = [x[1] for x in train]
X_test_raw = [x[0] for x in test]
y_test = [x[1] for x in test]

cv = CountVectorizer()
X_train_bow = cv.fit_transform(X_train_raw)
X_test_bow = cv.transform(X_test_raw)

nb = MultinomialNB(alpha=1.0)
nb.fit(X_train_bow, y_train)

priors = {cls: math.exp(lp) for cls, lp in zip(nb.classes_, nb.class_log_prior_)}
print("Class Priors:", priors)

nb_preds = nb.predict(X_test_bow)
nb_probs = nb.predict_proba(X_test_bow)

print("\nTest Set Predictions (Naive Bayes):")
for review, pred, prob in zip(X_test_raw, nb_preds, nb_probs):
    print(f"Review: '{review}' -> Predicted: {pred} (Probability: {max(prob):.4f})")

Class Priors: {np.str_('Negative'): 0.5000000000000001, np.str_('Positive'): 0.5000000000000001}

Test Set Predictions (Naive Bayes):
Review: 'Fresh food and fast delivery' -> Predicted: Positive (Probability: 0.8446)
Review: 'Late delivery and cold food' -> Predicted: Negative (Probability: 0.7465)
Review: 'Excellent taste, will order again' -> Predicted: Positive (Probability: 0.9354)
Review: 'Rude rider and terrible service' -> Predicted: Negative (Probability: 0.8548)


# **Part 3**

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train_raw)
X_test_tfidf = tfidf.transform(X_test_raw)

lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)

lr_preds = lr.predict(X_test_tfidf)
print(f"Logistic Regression Test Accuracy: {accuracy_score(y_test, lr_preds) * 100:.2f}%")

feature_names = tfidf.get_feature_names_out()
coefs = lr.coef_[0]
sorted_words = sorted(zip(feature_names, coefs), key=lambda x: x[1])

print("\nTop 3 Most Positive Words:", [(w, round(c, 4)) for w, c in sorted_words[-3:][::-1]])
print("Top 3 Most Negative Words:", [(w, round(c, 4)) for w, c in sorted_words[:3]])

print("\nPrediction Comparison (NB vs LR):")
for review, nb_p, lr_p in zip(X_test_raw, nb_preds, lr_preds):
    print(f"  '{review}' -> NB: {nb_p}, LR: {lr_p}")

Logistic Regression Test Accuracy: 100.00%

Top 3 Most Positive Words: [('fresh', np.float64(0.2171)), ('delicious', np.float64(0.2171)), ('loved', np.float64(0.2052))]
Top 3 Most Negative Words: [('was', np.float64(-0.2882)), ('late', np.float64(-0.2563)), ('cold', np.float64(-0.214))]

Prediction Comparison (NB vs LR):
  'Fresh food and fast delivery' -> NB: Positive, LR: Positive
  'Late delivery and cold food' -> NB: Negative, LR: Negative
  'Excellent taste, will order again' -> NB: Positive, LR: Positive
  'Rude rider and terrible service' -> NB: Negative, LR: Negative
